# Proyecto de regresión logística

En este notebook voy a hacer el ejercicio paso a paso. La idea es intentar predecir si un cliente del banco va a contratar o no un depósito a largo plazo.

## 1. Cargar los datos

Primero cargo el dataset para ver qué columnas trae y hacerme una idea general. No voy a hacer un análisis súper profundo, pero sí lo necesario para poder entrenar el modelo.

In [ ]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/4GeeksAcademy/logistic-regression-project-tutorial/main/bank-marketing-campaign-data.csv"
total_data = pd.read_csv(url, sep=";")

total_data.head()

In [ ]:
total_data.shape

## 2. Revisión rápida

Aquí miro si hay duplicados y si faltan datos. Antes de modelar prefiero dejar esto un poco limpio.

In [ ]:
total_data = total_data.drop_duplicates().reset_index(drop=True)
total_data.shape

In [ ]:
total_data.isnull().sum()

Veo que no hay nulos, así que en esta parte no hace falta rellenar nada. Ahora voy a transformar las columnas categóricas para poder trabajar con ellas.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Paso columnas de texto a números
total_data["job_n"] = pd.factorize(total_data["job"])[0]
total_data["marital_n"] = pd.factorize(total_data["marital"])[0]
total_data["education_n"] = pd.factorize(total_data["education"])[0]
total_data["default_n"] = pd.factorize(total_data["default"])[0]
total_data["housing_n"] = pd.factorize(total_data["housing"])[0]
total_data["loan_n"] = pd.factorize(total_data["loan"])[0]
total_data["contact_n"] = pd.factorize(total_data["contact"])[0]
total_data["month_n"] = pd.factorize(total_data["month"])[0]
total_data["day_of_week_n"] = pd.factorize(total_data["day_of_week"])[0]
total_data["poutcome_n"] = pd.factorize(total_data["poutcome"])[0]
total_data["y_n"] = pd.factorize(total_data["y"])[0]

num_variables = [
    "age", "job_n", "marital_n", "education_n", "default_n", "housing_n",
    "loan_n", "contact_n", "month_n", "day_of_week_n", "duration",
    "campaign", "pdays", "previous", "poutcome_n", "emp.var.rate",
    "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed", "y_n"
]

scaler = MinMaxScaler()
total_data_scal = pd.DataFrame(scaler.fit_transform(total_data[num_variables]), columns=num_variables)

total_data_scal.head()

## 3. Selección de variables

Para no quedarme con todas las columnas, voy a usar `SelectKBest` y me quedo con algunas de las más útiles para una primera versión del modelo.

In [ ]:
from sklearn.feature_selection import chi2, SelectKBest
from sklearn.model_selection import train_test_split

X = total_data_scal.drop("y_n", axis=1)
y = total_data_scal["y_n"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42
)

selection_model = SelectKBest(score_func=chi2, k=5)
selection_model.fit(X_train, y_train)

selected_features = X_train.columns[selection_model.get_support()]
selected_features

In [ ]:
X_train_sel = pd.DataFrame(selection_model.transform(X_train), columns=selected_features)
X_test_sel = pd.DataFrame(selection_model.transform(X_test), columns=selected_features)

X_train_sel.head()

Guardo también una copia de train y test por si luego quiero reutilizarlo sin repetir todo el proceso.

In [ ]:
import os

X_train_sel["y_n"] = list(y_train)
X_test_sel["y_n"] = list(y_test)

os.makedirs("../data/processed", exist_ok=True)

X_train_sel.to_csv("../data/processed/clean_train.csv", index=False)
X_test_sel.to_csv("../data/processed/clean_test.csv", index=False)

## 4. Modelo de regresión logística

Ahora entreno el modelo base, sin tocar todavía hiperparámetros, para ver qué resultado da de primeras.

In [ ]:
train_data = pd.read_csv("../data/processed/clean_train.csv")
test_data = pd.read_csv("../data/processed/clean_test.csv")

train_data.head()

In [ ]:
X_train = train_data.drop("y_n", axis=1)
y_train = train_data["y_n"]

X_test = test_data.drop("y_n", axis=1)
y_test = test_data["y_n"]

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_pred

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy_score(y_test, y_pred)

No me quedo solo con el accuracy, también quiero ver un poco mejor cómo se comporta el modelo.

In [ ]:
confusion_matrix(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

## 5. Pequeña optimización

Aquí pruebo una búsqueda de hiperparámetros para ver si puedo mejorar algo el resultado. No quiero complicarlo mucho, solo probar varias combinaciones.

In [ ]:
from sklearn.model_selection import GridSearchCV
import warnings

warnings.filterwarnings("ignore")

hyperparams = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear", "saga"]
}

grid = GridSearchCV(
    LogisticRegression(),
    hyperparams,
    scoring="accuracy",
    cv=10
)

grid.fit(X_train, y_train)

grid.best_params_

In [ ]:
best_model = grid.best_estimator_
best_model.fit(X_train, y_train)

y_pred_opt = best_model.predict(X_test)

accuracy_score(y_test, y_pred_opt)

In [ ]:
print(confusion_matrix(y_test, y_pred_opt))
print(classification_report(y_test, y_pred_opt))

## 6. Guardar el modelo

Para terminar guardo el modelo final. Así, si luego quisiera usarlo en otro archivo, no tendría que volver a entrenarlo.

In [ ]:
from pickle import dump

os.makedirs("../models", exist_ok=True)

dump(best_model, open("../models/logistic_regression_model.sav", "wb"))

## Conclusión

En resumen, he cargado los datos, he limpiado duplicados, he transformado las variables categóricas, he seleccionado algunas columnas, y después he entrenado una regresión logística. Luego probé una pequeña optimización con `GridSearchCV` para quedarme con una versión algo mejor del modelo.